# YgoMaster-PackBuilder — Policies Loader Tests

Notebook di test manuale/esplorativo per il **policies loader**.

Obiettivi:
- individuare la root della repository;
- caricare `config/policies.yaml`;
- verificare la costruzione dei DTO;
- controllare le distribuzioni di rarità;
- controllare i booleani YAML;
- verificare alcuni errori strutturali del loader.

## 1. Ambiente e root della repository

In [16]:
from pathlib import Path
import sys

print("Python interpreter:", sys.executable)
print("Current working directory:", Path.cwd())

cwd = Path.cwd().resolve()

if (cwd / "pyproject.toml").exists():
    repo_root = cwd
elif (cwd.parent / "pyproject.toml").exists():
    repo_root = cwd.parent
else:
    raise RuntimeError(
        "Impossibile individuare la root della repository. "
        "Esegui il notebook dalla root del progetto oppure dalla cartella tests/."
    )

policies_path = repo_root / "config" / "policies.yaml"

assert policies_path.exists(), f"Policies config non trovato: {policies_path}"

print("Repository root:", repo_root)
print("Policies path:", policies_path)

Python interpreter: c:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\.venv\Scripts\python.exe
Current working directory: c:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\tests
Repository root: C:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder
Policies path: C:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\config\policies.yaml


## 2. Import

In [17]:
from ygomaster_packbuilder.policies import (
    PoliciesConfig,
    load_policies_config,
)

from ygomaster_packbuilder.policies.models import (
    PolicyDefinition,
    PolicyRarityDistribution,
    PolicyTenPackGuarantee,
    PolicyUltraRarePity,
)

print("Policies imports OK")

Policies imports OK


## 3. Caricamento di `config/policies.yaml`

In [18]:
config = load_policies_config(policies_path)

assert isinstance(config, PoliciesConfig)

print("Policies loaded successfully")
config

Policies loaded successfully


PoliciesConfig(policies=[PolicyDefinition(key='configuration-a', normal_slot_distribution=PolicyRarityDistribution(normal=55.0, rare=35.0, super_rare=7.5, ultra_rare=2.5), ten_pack_guarantee=PolicyTenPackGuarantee(enabled=True, final_slot_distribution=PolicyRarityDistribution(normal=0.0, rare=0.0, super_rare=80.0, ultra_rare=20.0)), ultra_rare_pity=PolicyUltraRarePity(enabled=True, trigger_after_ten_pack_without_ur=True, guaranteed_slot_distribution=PolicyRarityDistribution(normal=0.0, rare=0.0, super_rare=0.0, ultra_rare=100.0)))])

## 4. Controllo della lista delle policy

In [19]:
assert isinstance(config.policies, list)
assert len(config.policies) > 0, "Nessuna policy trovata."

for policy in config.policies:
    assert isinstance(policy, PolicyDefinition)
    assert isinstance(policy.key, str)
    assert policy.key.strip()

print("Loaded policy keys:", [policy.key for policy in config.policies])

Loaded policy keys: ['configuration-a']


## 5. Controllo delle distribuzioni

In [20]:
def check_distribution(distribution: PolicyRarityDistribution) -> None:
    assert isinstance(distribution, PolicyRarityDistribution)
    assert isinstance(distribution.normal, float)
    assert isinstance(distribution.rare, float)
    assert isinstance(distribution.super_rare, float)
    assert isinstance(distribution.ultra_rare, float)

for policy in config.policies:
    check_distribution(policy.normal_slot_distribution)
    check_distribution(policy.ten_pack_guarantee.final_slot_distribution)
    check_distribution(policy.ultra_rare_pity.guaranteed_slot_distribution)

print("All rarity distributions loaded as floats")

All rarity distributions loaded as floats


## 6. Controllo dei booleani

In [21]:
for policy in config.policies:
    ten_pack = policy.ten_pack_guarantee
    pity = policy.ultra_rare_pity

    assert isinstance(ten_pack, PolicyTenPackGuarantee)
    assert isinstance(ten_pack.enabled, bool)

    assert isinstance(pity, PolicyUltraRarePity)
    assert isinstance(pity.enabled, bool)
    assert isinstance(pity.trigger_after_ten_pack_without_ur, bool)

print("Boolean fields OK")

Boolean fields OK


## 7. Valori attesi della Configuration A

In [22]:
configuration_a = next(
    (policy for policy in config.policies if policy.key == "configuration-a"),
    None,
)

assert configuration_a is not None, (
    "La policy 'configuration-a' non è presente in policies.yaml."
)

print(configuration_a)

normal = configuration_a.normal_slot_distribution

assert normal.normal == 55.0
assert normal.rare == 35.0
assert normal.super_rare == 7.5
assert normal.ultra_rare == 2.5

ten_pack = configuration_a.ten_pack_guarantee
assert ten_pack.enabled is True
assert ten_pack.final_slot_distribution.normal == 0.0
assert ten_pack.final_slot_distribution.rare == 0.0
assert ten_pack.final_slot_distribution.super_rare == 80.0
assert ten_pack.final_slot_distribution.ultra_rare == 20.0

pity = configuration_a.ultra_rare_pity
assert pity.enabled is True
assert pity.trigger_after_ten_pack_without_ur is True
assert pity.guaranteed_slot_distribution.normal == 0.0
assert pity.guaranteed_slot_distribution.rare == 0.0
assert pity.guaranteed_slot_distribution.super_rare == 0.0
assert pity.guaranteed_slot_distribution.ultra_rare == 100.0

print("Configuration A values OK")

PolicyDefinition(key='configuration-a', normal_slot_distribution=PolicyRarityDistribution(normal=55.0, rare=35.0, super_rare=7.5, ultra_rare=2.5), ten_pack_guarantee=PolicyTenPackGuarantee(enabled=True, final_slot_distribution=PolicyRarityDistribution(normal=0.0, rare=0.0, super_rare=80.0, ultra_rare=20.0)), ultra_rare_pity=PolicyUltraRarePity(enabled=True, trigger_after_ten_pack_without_ur=True, guaranteed_slot_distribution=PolicyRarityDistribution(normal=0.0, rare=0.0, super_rare=0.0, ultra_rare=100.0)))
Configuration A values OK


## 8. Test errore: file inesistente

In [23]:
missing_path = repo_root / "config" / "__policies_that_do_not_exist__.yaml"

try:
    load_policies_config(missing_path)
except FileNotFoundError as exc:
    print("Expected FileNotFoundError:", exc)
else:
    raise AssertionError("Il loader avrebbe dovuto sollevare FileNotFoundError.")

Expected FileNotFoundError: Policies file not found: C:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\config\__policies_that_do_not_exist__.yaml


## 9. Test errore: root YAML non mapping

In [24]:
from tempfile import TemporaryDirectory

with TemporaryDirectory() as tmpdir:
    invalid = Path(tmpdir) / "invalid_root.yaml"
    invalid.write_text("- first\n- second\n", encoding="utf-8")

    try:
        load_policies_config(invalid)
    except ValueError as exc:
        print("Expected ValueError:", exc)
    else:
        raise AssertionError("Il loader avrebbe dovuto rifiutare un root YAML non-mapping.")

Expected ValueError: Policies config root must be a mapping.


## 10. Test errore: `policies` non è una lista

In [25]:
with TemporaryDirectory() as tmpdir:
    invalid = Path(tmpdir) / "invalid_policies.yaml"
    invalid.write_text(
        'policies:\n  key: "configuration-a"\n',
        encoding="utf-8",
    )

    try:
        load_policies_config(invalid)
    except ValueError as exc:
        print("Expected ValueError:", exc)
    else:
        raise AssertionError("Il loader avrebbe dovuto rifiutare un campo 'policies' non-list.")

Expected ValueError: Invalid value for 'policies' in policies config: {'key': 'configuration-a'}


## 11. Test errore: booleano scritto come stringa

`enabled: false` è valido, mentre `enabled: "false"` deve essere rifiutato.


In [26]:
with TemporaryDirectory() as tmpdir:
    invalid = Path(tmpdir) / "invalid_bool.yaml"
    invalid.write_text(
        '''
policies:
  - key: "configuration-a"
    normal_slot_distribution:
      normal: 55
      rare: 35
      super_rare: 7.5
      ultra_rare: 2.5
    ten_pack_guarantee:
      enabled: "false"
      final_slot_distribution:
        normal: 0
        rare: 0
        super_rare: 80
        ultra_rare: 20
    ultra_rare_pity:
      enabled: true
      trigger_after_ten_pack_without_ur: true
      guaranteed_slot_distribution:
        normal: 0
        rare: 0
        super_rare: 0
        ultra_rare: 100
'''.strip(),
        encoding="utf-8",
    )

    try:
        load_policies_config(invalid)
    except ValueError as exc:
        print("Expected ValueError:", exc)
    else:
        raise AssertionError("Il loader avrebbe dovuto rifiutare enabled='false' come stringa.")

Expected ValueError: Invalid value for 'enabled' in ten pack guarantee: false


## 12. Test errore: pity trigger non booleano

In [27]:
with TemporaryDirectory() as tmpdir:
    invalid = Path(tmpdir) / "invalid_trigger_bool.yaml"
    invalid.write_text(
        '''
policies:
  - key: "configuration-a"
    normal_slot_distribution:
      normal: 55
      rare: 35
      super_rare: 7.5
      ultra_rare: 2.5
    ten_pack_guarantee:
      enabled: true
      final_slot_distribution:
        normal: 0
        rare: 0
        super_rare: 80
        ultra_rare: 20
    ultra_rare_pity:
      enabled: true
      trigger_after_ten_pack_without_ur: "true"
      guaranteed_slot_distribution:
        normal: 0
        rare: 0
        super_rare: 0
        ultra_rare: 100
'''.strip(),
        encoding="utf-8",
    )

    try:
        load_policies_config(invalid)
    except ValueError as exc:
        print("Expected ValueError:", exc)
    else:
        raise AssertionError(
            "Il loader avrebbe dovuto rifiutare "
            "trigger_after_ten_pack_without_ur='true' come stringa."
        )

Expected ValueError: Invalid value for 'trigger_after_ten_pack_without_ur' in ultra rare pity: true


## 13. Test conversione numerica

In [28]:
with TemporaryDirectory() as tmpdir:
    valid = Path(tmpdir) / "numeric_conversion.yaml"
    valid.write_text(
        '''
policies:
  - key: "test-policy"
    normal_slot_distribution:
      normal: 55
      rare: 35
      super_rare: 7.5
      ultra_rare: 2.5
    ten_pack_guarantee:
      enabled: true
      final_slot_distribution:
        normal: 0
        rare: 0
        super_rare: 80
        ultra_rare: 20
    ultra_rare_pity:
      enabled: true
      trigger_after_ten_pack_without_ur: true
      guaranteed_slot_distribution:
        normal: 0
        rare: 0
        super_rare: 0
        ultra_rare: 100
'''.strip(),
        encoding="utf-8",
    )

    loaded = load_policies_config(valid)
    distribution = loaded.policies[0].normal_slot_distribution

    assert isinstance(distribution.normal, float)
    assert isinstance(distribution.rare, float)
    assert isinstance(distribution.super_rare, float)
    assert isinstance(distribution.ultra_rare, float)

    print(distribution)

PolicyRarityDistribution(normal=55.0, rare=35.0, super_rare=7.5, ultra_rare=2.5)
